# ⚙️ Preprocesamiento y Feature Engineering
## Online Shoppers Purchasing Intention Dataset

**Objetivo**: Preparar los datos para modelado mediante limpieza, transformación y creación de nuevas features.

**Pasos**:
1. Limpieza de datos (duplicados, faltantes)
2. Feature Engineering (7 nuevas variables)
3. Encoding de variables categóricas
4. Manejo de outliers (winsorización)
5. Escalado de features
6. Balanceo de clases (SMOTE)
7. División train/test

---

## 1️⃣ Importar Librerías

In [36]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Preprocesamiento
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# Balanceo de clases
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

# Serialización
import joblib

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


## 2️⃣ Cargar Datos

In [37]:
# Cargar dataset original
df = pd.read_csv('../data/01_raw/online_shoppers_intention.csv')

print(f"📊 Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

📊 Dataset cargado: 12,330 filas × 18 columnas


## 3️⃣ Limpieza de Datos

In [38]:
# Verificar y eliminar duplicados
duplicates = df.duplicated().sum()
if duplicates > 0:
    df = df.drop_duplicates()
    print(f"✅ Eliminados {duplicates} duplicados")
else:
    print("✅ No hay duplicados")

# Verificar valores faltantes
missing = df.isnull().sum().sum()
print(f"✅ Valores faltantes: {missing}")

print(f"\n📊 Dataset final: {df.shape[0]:,} filas × {df.shape[1]} columnas")

✅ Eliminados 125 duplicados
✅ Valores faltantes: 0

📊 Dataset final: 12,205 filas × 18 columnas


## 4️⃣ Feature Engineering

Creamos 7 nuevas variables para capturar patrones de comportamiento:
- **TotalPages**: Suma total de páginas visitadas
- **TotalDuration**: Tiempo total en el sitio
- **AvgPageDuration**: Tiempo promedio por página
- **ProductRatio**: Proporción de páginas de producto
- **EngagementScore**: Combinación de métricas de engagement
- **IsShortSession**: Flag de sesión <30 segundos
- **IsHighInteraction**: Flag de >10 páginas visitadas

In [35]:
# Total de páginas visitadas en la sesión
df['TotalPages'] = df['Administrative'] + df['Informational'] + df['ProductRelated']

# Tiempo total en el sitio (suma de duraciones)
df['TotalDuration'] = (df['Administrative_Duration'] + 
                        df['Informational_Duration'] + 
                        df['ProductRelated_Duration'])

# Tiempo promedio por página (evitamos división por cero con +1)
df['AvgPageDuration'] = df['TotalDuration'] / (df['TotalPages'] + 1)

# Proporción de páginas de producto sobre total
df['ProductRatio'] = df['ProductRelated'] / (df['TotalPages'] + 1)

# Score de engagement: combina PageValues con tasas de interacción
# Valores altos = mayor engagement
df['EngagementScore'] = (df['PageValues'] * (1 - df['BounceRates']) * 
                          (1 - df['ExitRates']))

# Flag: sesión muy corta (menos de 30 segundos)
# Sesiones cortas suelen indicar menor intención de compra
df['IsShortSession'] = (df['TotalDuration'] < 30).astype(int)

# Flag: alta interacción (más de 10 páginas)
# Mayor exploración puede indicar mayor interés
df['IsHighInteraction'] = (df['TotalPages'] > 10).astype(int)

print("✅ Feature Engineering completado:")
new_features = ['TotalPages', 'TotalDuration', 'AvgPageDuration', 'ProductRatio', 
                'EngagementScore', 'IsShortSession', 'IsHighInteraction']
for i, feat in enumerate(new_features, 1):
    print(f"  {i}. {feat}")

✅ Feature Engineering completado:
  1. TotalPages
  2. TotalDuration
  3. AvgPageDuration
  4. ProductRatio
  5. EngagementScore
  6. IsShortSession
  7. IsHighInteraction


## 5️⃣ Manejo de Outliers

**Estrategia**: Winsorización en lugar de eliminación
- Mantiene el tamaño del dataset
- Preserva información de sesiones extremas
- Capea valores en percentiles 1 y 99

In [22]:
# Variables susceptibles a outliers extremos
numeric_cols = ['Administrative_Duration', 'Informational_Duration', 
                'ProductRelated_Duration', 'BounceRates', 'ExitRates', 
                'PageValues', 'TotalDuration', 'AvgPageDuration']

# Winsorización: capear valores en percentiles 1 y 99
# Evita que valores extremos distorsionen el escalado
from scipy.stats.mstats import winsorize

for col in numeric_cols:
    df[col] = winsorize(df[col], limits=[0.01, 0.01])
    
print(f"✅ Outliers tratados en {len(numeric_cols)} variables mediante winsorización")

✅ Outliers tratados en 8 variables mediante winsorización


## 6️⃣ Encoding de Variables Categóricas

In [23]:
# Encoding de variables categóricas

# 1. Month: Ordinal Encoding (preserva orden temporal)
month_mapping = {
    'Jan': 1, 'Feb': 2, 'Mar': 3, 'Apr': 4, 'May': 5, 'Jun': 6,
    'Jul': 7, 'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12
}
df['Month'] = df['Month'].map(month_mapping)
print("✅ Month: Ordinal encoding (1-12)")

# 2. VisitorType: Label Encoding
visitor_mapping = {
    'New_Visitor': 0,
    'Returning_Visitor': 1,
    'Other': 2
}
df['VisitorType'] = df['VisitorType'].map(visitor_mapping)
print("✅ VisitorType: Label encoding (0=New, 1=Returning, 2=Other)")

# 3. Weekend: Boolean a int
df['Weekend'] = df['Weekend'].astype(int)
print("✅ Weekend: Boolean a int (0/1)")

# 4. Revenue: Boolean a int (target variable)
df['Revenue'] = df['Revenue'].astype(int)
print("✅ Revenue: Boolean a int (0=No Compra, 1=Compra)")

# Variables técnicas (OperatingSystems, Browser, Region, TrafficType) 
# ya están en formato numérico desde el dataset original
print("\n📊 Variables categóricas codificadas correctamente")

✅ Month: Ordinal encoding (1-12)
✅ VisitorType: Label encoding (0=New, 1=Returning, 2=Other)
✅ Weekend: Boolean a int (0/1)
✅ Revenue: Boolean a int (0=No Compra, 1=Compra)

📊 Variables categóricas codificadas correctamente


## 7️⃣ Selección de Features para Modelado

In [25]:
# Definir features finales
feature_columns = [
    # Métricas originales
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
    
    # Variables categóricas codificadas
    'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend',
    
    # Features creadas
    'TotalPages', 'TotalDuration', 'AvgPageDuration',
    'ProductRatio', 'EngagementScore', 'IsShortSession', 'IsHighInteraction'
]

target_column = 'Revenue'

X = df[feature_columns].copy()
y = df[target_column].copy()

# Manejar NaN que pueden venir del feature engineering
X = X.fillna(0)

print(f"✅ Features seleccionadas: {len(feature_columns)}")
print(f"📊 Dataset para modelado: {X.shape}")
print(f"🎯 Target distribution:")
print(f"  - No Compra (0): {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.2f}%)")
print(f"  - Compra (1): {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.2f}%)")
print(f"  - NaN en X: {X.isna().sum().sum()}")
print(f"  - NaN en y: {y.isna().sum()}")

✅ Features seleccionadas: 24
📊 Dataset para modelado: (12205, 24)
🎯 Target distribution:
  - No Compra (0): 10,297 (84.37%)
  - Compra (1): 1,908 (15.63%)
  - NaN en X: 0
  - NaN en y: 0


## 8️⃣ Escalado de Features

Estandarizamos las variables numéricas para mejorar el rendimiento del modelo:

In [26]:
# Inicializar scaler
scaler = StandardScaler()

# Aplicar escalado
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_columns, index=X.index)

print("✅ Features escaladas usando StandardScaler")
print(f"📊 Media después del escalado: {X_scaled.mean().mean():.4f}")
print(f"📊 Desviación estándar: {X_scaled.std().mean():.4f}")

✅ Features escaladas usando StandardScaler
📊 Media después del escalado: -0.0000
📊 Desviación estándar: 1.0000


## 7️⃣ División de Datos (Train/Test)

**Estrategia**:
- Test size: 20% (2,466 muestras)
- Estratificado por Revenue (mantiene proporción de clases)
- Random state: 42 (reproducibilidad)

In [27]:
# División estratificada: mantiene proporción de clases en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.2,      # 20% para test
    random_state=42,    # Reproducibilidad
    stratify=y          # Mantener proporción de clases
)

print(f"✅ Datos divididos:")
print(f"  Train: {X_train.shape[0]:,} muestras ({y_train.sum()/len(y_train)*100:.1f}% Compra)")
print(f"  Test:  {X_test.shape[0]:,} muestras ({y_test.sum()/len(y_test)*100:.1f}% Compra)")

✅ Datos divididos:
  Train: 9,764 muestras (15.6% Compra)
  Test:  2,441 muestras (15.6% Compra)


## 8️⃣ Balanceo de Clases con SMOTE

**Problema**: 84.5% No Compra vs 15.5% Compra (ratio 5.4:1)  
**Solución**: SMOTE (Synthetic Minority Over-sampling Technique)
- Genera muestras sintéticas de la clase minoritaria
- Solo se aplica al conjunto de entrenamiento
- Test mantiene distribución real para evaluación honesta

In [28]:
# SMOTE: genera muestras sintéticas de la clase minoritaria
smote = SMOTE(random_state=42)

print("⚖️ Distribución ANTES de SMOTE:")
print(f"  Clase 0 (No Compra): {(y_train==0).sum():,} ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"  Clase 1 (Compra):    {(y_train==1).sum():,} ({(y_train==1).sum()/len(y_train)*100:.1f}%)")

# Aplicar SMOTE solo a train
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\n⚖️ Distribución DESPUÉS de SMOTE:")
print(f"  Clase 0 (No Compra): {(y_train_balanced==0).sum():,} ({(y_train_balanced==0).sum()/len(y_train_balanced)*100:.1f}%)")
print(f"  Clase 1 (Compra):    {(y_train_balanced==1).sum():,} ({(y_train_balanced==1).sum()/len(y_train_balanced)*100:.1f}%)")
print(f"\n✅ Dataset balanceado: {len(y_train):,} → {len(y_train_balanced):,} muestras")

⚖️ Distribución ANTES de SMOTE:
  Clase 0 (No Compra): 8,238 (84.4%)
  Clase 1 (Compra):    1,526 (15.6%)

⚖️ Distribución DESPUÉS de SMOTE:
  Clase 0 (No Compra): 8,238 (50.0%)
  Clase 1 (Compra):    8,238 (50.0%)

✅ Dataset balanceado: 9,764 → 16,476 muestras


## 1️⃣1️⃣ Guardar Datos Procesados

In [29]:
# Crear dataset procesado completo
df_processed = df.copy()
df_processed[feature_columns] = X_scaled

# Guardar dataset procesado
df_processed.to_csv('../data/02_processed/processed_shoppers_data.csv', index=False)
print("✅ Dataset procesado guardado en data/02_processed/")

# Guardar scaler para uso en producción
joblib.dump(scaler, '../models/scaler.pkl')
print("✅ Scaler guardado en models/scaler.pkl")

# Guardar conjuntos de entrenamiento y prueba
np.save('../data/02_processed/X_train.npy', X_train)
np.save('../data/02_processed/X_test.npy', X_test)
np.save('../data/02_processed/y_train.npy', y_train)
np.save('../data/02_processed/y_test.npy', y_test)
np.save('../data/02_processed/X_train_balanced.npy', X_train_balanced)
np.save('../data/02_processed/y_train_balanced.npy', y_train_balanced)

print("✅ Conjuntos train/test guardados en formato numpy")

✅ Dataset procesado guardado en data/02_processed/
✅ Scaler guardado en models/scaler.pkl
✅ Conjuntos train/test guardados en formato numpy


In [30]:
# Guardar lista de features para referencia
with open('../data/02_processed/feature_names.txt', 'w') as f:
    for feature in feature_columns:
        f.write(f"{feature}\n")

print("✅ Lista de features guardada para referencia")
print(f"\n📋 Total de features: {len(feature_columns)}")

✅ Lista de features guardada para referencia

📋 Total de features: 24


## 1️⃣2️⃣ Resumen del Preprocesamiento

In [31]:
print("="*80)
print("📝 RESUMEN DEL PREPROCESAMIENTO")
print("="*80)

print("\n✅ DATOS LIMPIOS:")
print(f"  - Duplicados eliminados: {duplicates}")
print(f"  - Valores faltantes: {missing}")
print(f"  - Filas finales: {df.shape[0]:,}")

print("\n🔧 FEATURE ENGINEERING:")
print(f"  - Features originales: 17")
print(f"  - Features creadas: 7")
print(f"  - Features totales: {len(feature_columns)}")

print("\n⚖️ BALANCEO DE CLASES:")
print(f"  - Método: SMOTE")
print(f"  - Train original: {len(y_train):,} muestras")
print(f"  - Train balanceado: {len(y_train_balanced):,} muestras")
print(f"  - Test (sin balancear): {len(y_test):,} muestras")

print("\n📊 TRANSFORMACIONES:")
print("  ✅ Encoding de variables categóricas")
print("  ✅ Winsorización de outliers")
print("  ✅ Estandarización con StandardScaler")
print("  ✅ Balanceo con SMOTE")

print("\n💾 ARCHIVOS GUARDADOS:")
print("  ✅ processed_shoppers_data.csv")
print("  ✅ scaler.pkl")
print("  ✅ X_train.npy, X_test.npy")
print("  ✅ y_train.npy, y_test.npy")
print("  ✅ X_train_balanced.npy, y_train_balanced.npy")
print("  ✅ feature_names.txt")

print("\n🚀 SIGUIENTE PASO:")
print("  📓 Notebook 03_modelado_dataset.ipynb - Entrenamiento de modelos ML")
print("="*80)

📝 RESUMEN DEL PREPROCESAMIENTO

✅ DATOS LIMPIOS:
  - Duplicados eliminados: 125
  - Valores faltantes: 0
  - Filas finales: 12,205

🔧 FEATURE ENGINEERING:
  - Features originales: 17
  - Features creadas: 7
  - Features totales: 24

⚖️ BALANCEO DE CLASES:
  - Método: SMOTE
  - Train original: 9,764 muestras
  - Train balanceado: 16,476 muestras
  - Test (sin balancear): 2,441 muestras

📊 TRANSFORMACIONES:
  ✅ Encoding de variables categóricas
  ✅ Winsorización de outliers
  ✅ Estandarización con StandardScaler
  ✅ Balanceo con SMOTE

💾 ARCHIVOS GUARDADOS:
  ✅ processed_shoppers_data.csv
  ✅ scaler.pkl
  ✅ X_train.npy, X_test.npy
  ✅ y_train.npy, y_test.npy
  ✅ X_train_balanced.npy, y_train_balanced.npy
  ✅ feature_names.txt

🚀 SIGUIENTE PASO:
  📓 Notebook 03_modelado_dataset.ipynb - Entrenamiento de modelos ML


---

## ✅ Resumen del Preprocesamiento

### 🔧 Transformaciones Aplicadas

1. **Limpieza de Datos**
   - ✅ Verificación de duplicados: 0 encontrados
   - ✅ Verificación de faltantes: 0 encontrados
   - ✅ Dataset limpio y listo

2. **Feature Engineering (7 nuevas variables)**
   - `TotalPages`: Suma de todas las páginas visitadas
   - `TotalDuration`: Tiempo total en el sitio
   - `AvgPageDuration`: Tiempo promedio por página
   - `ProductRatio`: Proporción de páginas de productos
   - `EngagementScore`: Score combinado de engagement
   - `IsShortSession`: Flag de sesión <30s
   - `IsHighInteraction`: Flag de >10 páginas

3. **Manejo de Outliers**
   - Método: Winsorización (percentiles 1-99)
   - Variables tratadas: Duraciones, BounceRates, ExitRates, PageValues
   - Beneficio: Mantiene tamaño del dataset, evita valores extremos

4. **Encoding de Categóricas**
   - `Month`: Ordinal (1-12) para preservar estacionalidad
   - `VisitorType`: Label (0=New, 1=Returning, 2=Other)
   - `Weekend`: Boolean → int (0/1)
   - `Revenue`: Boolean → int (0/1)

5. **Escalado**
   - StandardScaler: Media=0, Desviación=1
   - Guardado en: `../models/scaler.pkl`

6. **Balanceo de Clases**
   - Método: SMOTE (Synthetic Minority Over-sampling)
   - Train antes: 9,864 muestras (84.5% / 15.5%)
   - Train después: 16,688 muestras (50% / 50%)
   - Test: Sin modificar (distribución real)

### 📊 Dataset Final

| Conjunto | Muestras | Balance | Features |
|----------|----------|---------|----------|
| **Train (Original)** | 9,864 | 84.5% / 15.5% | 24 |
| **Train (Balanceado)** | 16,688 | 50% / 50% | 24 |
| **Test** | 2,466 | 84.5% / 15.5% | 24 |

### 💾 Archivos Generados

```
data/02_processed/
├── X_train.npy              # Train original (9,864 × 24)
├── y_train.npy              # Target train original
├── X_train_balanced.npy     # Train balanceado SMOTE (16,688 × 24)
├── y_train_balanced.npy     # Target train balanceado
├── X_test.npy               # Test (2,466 × 24)
├── y_test.npy               # Target test
└── feature_names.txt        # Lista de 24 features

models/
└── scaler.pkl               # StandardScaler entrenado
```

### 🚀 Siguiente Paso

**Notebook 03_modelado_dataset.ipynb** - Entrenar y evaluar modelos de clasificación:
- Logistic Regression, Decision Tree, Random Forest
- Gradient Boosting, XGBoost, LightGBM
- GridSearchCV para optimización
- Evaluación con múltiples métricas

---

**⚙️ Pipeline completo**: Limpieza → Feature Engineering → Encoding → Outliers → Escalado → Balanceo